# 01 Dataset Audit

Use this after stages 01-03 in the master notebook. This notebook proves that labels, participants, splits, and modality availability are usable before any audio features are trusted.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/home/ubuntu/nishn_workspce/test_pdfs_generic/.covid_audio_btp_private/covid_audio_btp")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from covid_audio_btp.notebook_utils import (
    PROJECT_ROOT,
    artifact,
    check_artifacts,
    count_table,
    read_csv,
    read_optional_csv,
    require_artifacts,
    save_table,
    value_counts_frame,
    assert_no_participant_leakage,
    assert_binary_labels_present,
    stop_if_validation_errors,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
print(PROJECT_ROOT)


In [ ]:
require_artifacts([
    "data/interim/coswara_index.csv",
    "data/processed/metadata_clean.csv",
    "data/interim/modality_availability.csv",
    "data/interim/split_manifest.csv",
])
index_df = read_csv("data/interim/coswara_index.csv", n=5)
metadata = read_csv("data/processed/metadata_clean.csv", n=5)
availability = read_csv("data/interim/modality_availability.csv", n=5)
split_manifest = read_csv("data/interim/split_manifest.csv", n=5)


## Hard Gates


In [ ]:
assert_binary_labels_present(metadata)
assert_no_participant_leakage(metadata)
assert_no_participant_leakage(split_manifest)

unknown_rate = (metadata["label_binary"] == "unknown").mean() if "label_binary" in metadata else 1.0
print(f"unknown label rate: {unknown_rate:.3f}")
if unknown_rate > 0.10:
    raise AssertionError("Too many unknown labels. Inspect label_raw values before training.")


## Dataset Tables


In [ ]:
label_by_modality = count_table(metadata, ["dataset", "modality", "label_binary"])
split_by_label = count_table(metadata, ["split", "label_binary"])
modality_by_split = count_table(metadata, ["split", "modality"])

save_table(label_by_modality, "reports/tables/nb01_label_by_modality.csv")
save_table(split_by_label, "reports/tables/nb01_split_by_label.csv")
save_table(modality_by_split, "reports/tables/nb01_modality_by_split.csv")

display(label_by_modality)
display(split_by_label)
display(modality_by_split)


## Modality Availability


In [ ]:
availability_summary = pd.DataFrame({
    "metric": ["participants", "has_cough", "has_breath", "has_speech", "complete_case"],
    "value": [
        len(availability),
        availability.get("has_cough", pd.Series(dtype=bool)).mean(),
        availability.get("has_breath", pd.Series(dtype=bool)).mean(),
        availability.get("has_speech", pd.Series(dtype=bool)).mean(),
        availability.get("complete_case", pd.Series(dtype=bool)).mean(),
    ],
})
save_table(availability_summary, "reports/tables/nb01_modality_availability_summary.csv")
display(availability_summary)
availability[[c for c in ["participant_id", "available_modalities", "complete_case", "n_cough", "n_breath", "n_speech"] if c in availability.columns]].head(20)


## Visual Checks


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=metadata, x="split", hue="label_binary", ax=axes[0])
axes[0].set_title("Class distribution by split")
sns.countplot(data=metadata, x="modality", hue="label_binary", ax=axes[1])
axes[1].set_title("Class distribution by modality")
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "reports/figures/nb01_dataset_distribution.png", dpi=160)
plt.show()


## Manual Quality Metadata Check

Coswara has human quality fields in some metadata versions. These are not the same as our computed audio quality audit, but they are useful for diagnosing bad recordings.


In [ ]:
manual_cols = [c for c in ["manual_quality_score", "manual_quality_label"] if c in metadata.columns]
if manual_cols:
    display(metadata[manual_cols + ["modality", "label_binary"]].head(20))
    display(count_table(metadata, ["modality", "manual_quality_label"]))
else:
    print("No manual quality columns found in metadata.")
